# PARITY — T4 GPU run book

Everything in this project that has **never executed on a GPU**, in one notebook.
Phases 4, 5, 2 and 6. Phases 1, 3, 7, 8 are CPU-only and already done.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
The notebook clones the repo itself; nothing needs uploading.

Expect roughly 30–45 min, plus ~20 min more if you enable the optional OpenACC-offload cell.

Ordering note: phase 2 runs **before** phase 6 on purpose. `bench.quant` needs the
nested seed-0 autoencoder checkpoint, so training has to happen first, and doing it in
this order means phase 6 scores the T4-trained latents rather than a stale cache.

What each phase is for, and what to watch:

| Phase | Command | The thing that could go wrong |
|---|---|---|
| 4 | CUDA + OpenACC allocators | bit-identity vs the serial oracle breaks under real scheduling |
| 5 | ordering sweep | modelled time was never measured; this settles the Pareto question |
| 2 | latent retrain | spec always said T4; has only ever run on CPU |
| 6 | INT8 tensor-core scoring | `gemm_bitexact` must be True; N=2000 is launch-bound |

## 0 — Confirm the GPU is a T4

Every occupancy number in `STATE.md` is derived against **sm_75, 40 SMs, 16 GB**.
If Colab hands you a different card the numbers still run but stop being comparable.

In [ ]:
!nvidia-smi

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", out)
if "T4" not in out:
    print("\n!! NOT a T4. Occupancy claims in STATE.md are sm_75-specific.")
    print("   Runtime -> Change runtime type -> T4 GPU, then Run all again.")
else:
    print("OK - sm_75 as assumed by STATE.md.")

## 1 — Clone the repo and install what is missing

Colab already has torch with CUDA. Missing here: `numba` (phases 4–6) and
`sentence-transformers` (phase 2).

`latent/cache/` is gitignored so it does **not** come with the clone — phase 2 rebuilds it
from `latent/data/corpus.jsonl`, which *is* committed. Nothing needs uploading.

In [ ]:
import os

REPO = "/content/cgvr"
if os.path.isdir(REPO):
    # a reused Colab session keeps the old clone - pull, or you silently run stale code
    !git -C {REPO} pull -q --ff-only
else:
    !git clone -q https://github.com/chhavi-pareek/cgvr {REPO}

os.chdir(REPO)
!git log --oneline -1

missing = [p for p in ("alloc/cuda/allocator.py", "order/sweep.py",
                       "tests/test_parallel.py", "latent/data/corpus.jsonl")
           if not os.path.exists(p)]
print("MISSING:", missing) if missing else print("source tree OK")

In [ ]:
!pip install -q numba sentence-transformers 2>&1 | tail -3

### Check numba actually sees the GPU

This is the single most important check in the notebook. `tests/conftest.py` and
`alloc/cuda/__init__.py` silently fall back to `NUMBA_ENABLE_CUDASIM=1` when no driver is
found — so a broken CUDA setup does **not** fail loudly, it just runs the simulator and
every phase-4 test passes without touching the GPU. That would make the whole run
worthless while looking like a success.

Recent numba moved the CUDA target into a separate `numba-cuda` package. If the check
below fails, run the fallback cell.

In [ ]:
import os
os.environ.pop("NUMBA_ENABLE_CUDASIM", None)   # never let the simulator stand in on the T4

import numba
from numba import cuda
print("numba", numba.__version__)
print("cuda.is_available():", cuda.is_available())
if cuda.is_available():
    d = cuda.get_current_device()
    print("device:", d.name.decode() if isinstance(d.name, bytes) else d.name,
          "| compute capability:", d.compute_capability)
    print("\nOK - real GPU, not the simulator.")
else:
    print("\n!! numba cannot see the GPU. Run the fallback cell below, then re-run this one.")

In [ ]:
# FALLBACK - only run this if the cell above said numba cannot see the GPU.
# Then re-run the cell above to confirm, and carry on.
!pip install -q numba-cuda 2>&1 | tail -3
print("Installed numba-cuda. Re-run the check cell above (you may need Runtime -> Restart session first).")

# Phase 4 — the parallel allocator

The CS372IA core. Four implementations, all of which must reproduce the serial oracle
**bit for bit**. Everything so far was validated under numba's CUDA *simulator*, which
does not model warp scheduling, so this is the first real test.

Two specific hazards recorded in `STATE.md`:

- **FMA contraction.** `score = (s*q) - (lam*c)` loses bit-identity if NVVM contracts it
  into an FMA. The CUDA path uses libdevice `dmul_rn`/`dadd_rn` to prevent that. If the
  tests fail on assignment mismatch, this is the first suspect.
- **The last-block-done pattern.** The device-side bisection advances when the final block
  finishes. Under the simulator, block order is trivial; on real hardware it is not.

**Preflight — read this if the tests fail to collect.**

`tests/test_parallel.py` does `from tests.test_alloc import ...`. Colab ships its own
top-level `tests` package in site-packages, and it shadows the repo's, so that import
resolves to the wrong package and collection dies with
`ModuleNotFoundError: No module named 'tests.test_alloc'`.

(The message names `tests.test_alloc`, not `tests` — that is the tell: the parent *was*
found, just not yours.)

The fix is one empty file. `tests/__init__.py` makes the repo's `tests` a real package,
so pytest puts the repo root first on `sys.path` and it wins outright. The cell below
creates it if the clone predates the fix, then proves the right one resolves before any
test runs.

In [ ]:
import os
import subprocess
import sys

REPO = "/content/cgvr"
init = os.path.join(REPO, "tests", "__init__.py")
if not os.path.exists(init):
    open(init, "w").close()
    print("created tests/__init__.py")
else:
    print("tests/__init__.py already present")

chk = subprocess.run(
    [sys.executable, "-c",
     "import tests, tests.test_alloc as t;"
     "print('tests resolved to:', tests.__file__);"
     "print('_theta imported:', t._theta is not None)"],
    cwd=REPO, capture_output=True, text=True)
print(chk.stdout or chk.stderr)
print("PREFLIGHT OK" if chk.returncode == 0 else "PREFLIGHT FAILED - stop and read the error above")

In [ ]:
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m pytest -q tests/test_parallel.py -k cuda 2>&1 | tail -20

### OpenACC — host build first

`make serial` builds a host-only library with `-ffp-contract=off`. That is the same
correctness check that passed on the Mac, repeated here so the oracle comparison is
anchored before any offload build.

In [ ]:
!cd /content/cgvr/alloc/openacc && make -s serial && echo "built libparity_acc.so (host)"
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m pytest -q tests/test_parallel.py -k openacc 2>&1 | tail -20

In [ ]:
from alloc.openacc import backend
print("OpenACC backend currently loaded:", backend())

### OPTIONAL — OpenACC offload to the T4

This needs the NVIDIA HPC SDK for `nvc`, which is a **large download (~10 GB) and takes
15–25 min**. Skip it unless you specifically want the GPU-offload OpenACC numbers; the
correctness result above already stands on the host build.

`STATE.md` flags two pragmas that have only ever been compiled as plain C and are
unverified under a real OpenACC compiler — watch the `-Minfo=accel` output for both:

- `#pragma acc update self(bj[besti:1], abuf[off+besti:1])` inside the fill
- `reduction(max:bestr)` with an `if` update (legal, but worth confirming it vectorises)

In [ ]:
# Uncomment the whole cell to install the HPC SDK (~10 GB, 15-25 min).
#
# !curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
#     | gpg --dearmor -o /usr/share/keyrings/nvhpc-archive-keyring.gpg
# !echo 'deb [signed-by=/usr/share/keyrings/nvhpc-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
#     > /etc/apt/sources.list.d/nvhpc.list
# !apt-get update -qq && apt-get install -y -qq nvhpc-24-5 2>&1 | tail -5
# import os, glob
# nvbin = sorted(glob.glob("/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin"))[-1]
# os.environ["PATH"] = nvbin + ":" + os.environ["PATH"]
# !nvc --version
print("Skipped by default - see the comment block in this cell.")

In [ ]:
# Run this only after the HPC SDK cell above succeeded.
# !cd /content/cgvr/alloc/openacc && make clean && make acc 2>&1 | tail -30
# !cd /content/cgvr && PYTHONPATH=/content/cgvr python -m pytest -q tests/test_parallel.py -k openacc 2>&1 | tail -20
# from alloc.openacc import backend; print("backend now:", backend())
print("Skipped by default.")

### Speedup, efficiency, Amdahl framing

`N=100000` has **never been run** — the Mac was capped at 10k during development.
This is the headline four-way comparison for CS372IA.

In [ ]:
# numba's set_num_threads refuses a count above the machine's cores, and Colab's free
# tier gives 2 - so build the thread list from what is actually there.
import os

nproc = os.cpu_count() or 2
thr = " ".join(str(t) for t in (1, 2, 4, 8) if t <= nproc) or "1"
print(f"{nproc} CPU cores -> --threads {thr}")

In [ ]:
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m bench.speedup --sizes 1000 10000 100000 --threads {thr}

### Occupancy, measured rather than analytic

Off-hardware this prints only the analytic table. Here it should also report
**measured regs/thread** and the driver's **active blocks/SM**.

`STATE.md` predicts: ≤64 regs/thread → 4 blocks/SM → 32/32 warps (100% occupancy);
65–80 regs → 3 blocks (75%). Check which side you land on.

In [ ]:
import os
import sys

os.chdir("/content/cgvr")
if "/content/cgvr" not in sys.path:
    sys.path.insert(0, "/content/cgvr")
import numpy as np

from alloc.config import build_table
from alloc.cuda.allocator import CudaAllocator, occupancy_report
from alloc.serial import SerialAllocator
from bench.speedup import instance

table = build_table()
alloc = CudaAllocator(table)

# occupancy_report reads k_eval.overloads, which stays empty until the kernel has been
# compiled AND launched - so do one real allocation first, or it reports analytic only.
s, cost = instance(0, 1024, table)
t0 = SerialAllocator(table).allocate(s, cost, np.inf).cost
alloc.allocate(s, cost, 0.5 * t0)

print(occupancy_report(alloc))

# Phase 5 — the ordering study

This phase decides an **open research question**. The phase 5 Pareto criterion
("pure-Morton and pure-state are each dominated somewhere") currently *fails* as stated.
It only holds in **modelled** time, with thin margins — 2–4% at sparse/mixed, 13% at dense —
and that model is two assumed constants (0.5 IPC, 1 TB/s), not measurement.

Real CUDA-event timings either rescue the criterion or settle that you need to change
the kernel. `t_model_us` stays in the CSV next to the measured time so you can compare.

**Watch `wee_meas == wee`.** The sweep raises if the instrumented in-kernel warp counters
disagree with the host-side model. That equality is the entire justification for claiming
warp execution efficiency without Nsight, so if it breaks, the phase 5 method breaks.

In [ ]:
# Calibration sweep: N = 50k (2000 agents x 5x5 tiles), 11 w values + blind, 3 densities.
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m order.sweep --agents 2000 --tile 5 --frames 3

In [ ]:
# Held-out sweep: different seed and sizes, for the adaptive policy.
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m order.sweep --seed 1 --tile 4 --agents 2000 1000 \
    --out bench/logs/order_sweep_test.csv

In [ ]:
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m order.adaptive 2>&1 | tail -25

In [ ]:
# Did the in-kernel counters agree with the host model, and is measured time close to modelled?
import csv
rows = list(csv.DictReader(open("/content/cgvr/bench/logs/order_sweep.csv")))
print("columns:", ", ".join(rows[0].keys()))

bad = [r for r in rows if r["wee_meas"] not in ("", "nan")
       and abs(float(r["wee_meas"]) - float(r["wee"])) > 1e-9]
print(f"\ncounter mismatches (wee_meas != wee): {len(bad)} of {len(rows)}")
print("timing source(s) in CSV:", {r["source"] for r in rows})

# t_meas_us is the CUDA-event median; t_model_us is the phase 5 assumption-based model.
print(f"\n{'density':9s} {'w':>5s} {'measured us':>12s} {'modelled us':>12s} {'meas/model':>11s}")
for r in rows:
    if r["w"] in ("0.0", "1.0") and r["t_meas_us"] not in ("", "nan"):
        m, mod = float(r["t_meas_us"]), float(r["t_model_us"])
        print(f"{r['density']:9s} {r['w']:>5s} {m:>12.2f} {mod:>12.2f} {m / mod:>11.2f}")

# The open question: at dense, does MEASURED time still prefer w=0 the way the model did?
print("\nbest measured w per density (this is the phase 5 Pareto question):")
for d in sorted({r["density"] for r in rows}):
    c = [r for r in rows if r["density"] == d and r["t_meas_us"] not in ("", "nan")]
    if c:
        b = min(c, key=lambda r: float(r["t_meas_us"]))
        print(f"  {d:9s} w={b['w']:>5s}  {float(b['t_meas_us']):.2f} us")

# Phase 2 — retrain the latent on the T4

The phase spec always said phase 2 runs on the T4; it has only ever run on CPU.
This trains 20 models (10 seeds x nested/plain) and regenerates the truncation table.

**An honest caveat:** `latent/train.py` never moves the model to CUDA — there is no
`.to(device)` in it — so the autoencoder training itself runs on the **CPU** even here.
Only `latent/embed.py` uses the GPU. So this run reproduces phase 2 on Colab hardware and
regenerates the cache the next phase needs, but it does not make phase 2 "a GPU result".
If you want that claim, `train.py` needs a device argument first.

Held-out nMSE at k=16 should land near **behaviour 0.132 / animation 0.196**.

In [ ]:
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m latent.embed 2>&1 | tail -3

In [ ]:
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m latent.train 2>&1 | tail -25

In [ ]:
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m latent.truncation 2>&1 | tail -30

# Phase 6 — INT8 tensor-core scoring

Two things to check:

1. **`gemm_bitexact` must be True on every row.** Both the `torch._int_mm` IMMA path and
   the NumPy INT32 reference are exact integer arithmetic, so any mismatch is a real bug,
   not rounding.
2. **N=2000 is launch-bound.** At N=2000, K=32, m=184 the GEMM is only ~12 MFLOP, so
   `gemm_us` vs `fp32_us` there says nothing about tensor cores. The second cell runs
   N=50000, which is where the comparison starts to mean something.

In [ ]:
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m bench.quant 2>&1 | tail -30

In [ ]:
# The run that can actually support a tensor-core claim.
!cd /content/cgvr && PYTHONPATH=/content/cgvr python -m bench.quant --agents 50000 --out bench/logs/quant_50k.csv 2>&1 | tail -30

In [ ]:
import csv
for name in ("quant.csv", "quant_50k.csv"):
    p = f"/content/cgvr/bench/logs/{name}"
    try:
        rows = list(csv.DictReader(open(p)))
    except FileNotFoundError:
        print(f"{name}: not produced"); continue
    bad = [r for r in rows if r["gemm_bitexact"].lower() not in ("true", "1")]
    print(f"\n{name}: {len(rows)} rows | gemm_bitexact False on {len(bad)}")
    print(f"  {'d':>3s} {'lam':>8s} {'provable':>9s} {'gemm_us':>9s} {'fp32_us':>9s} {'backend':>10s}")
    for r in rows:
        if r["interval"] == "pairwise":
            print(f"  {r['d']:>3s} {float(r['lam']):>8.2f} {float(r['provable']):>9.3f} "
                  f"{r['gemm_us']:>9s} {r['fp32_us']:>9s} {r['backend']:>10s}")

# Collect the results

Everything the figures need. Downloads `parity_t4_results.zip` (or copies it to Drive, if
you happen to have Drive mounted). Hand it back — the CSVs drop straight into
`bench/logs/` on the Mac and figure 4 stops being hollow.

In [ ]:
!cd /content/cgvr && ls -la bench/logs/

In [ ]:
import os
import shutil

!cd /content/cgvr && zip -q -r /content/parity_t4_results.zip bench/logs latent/data/truncation.json
print("wrote /content/parity_t4_results.zip",
      os.path.getsize("/content/parity_t4_results.zip"), "bytes")

# Copy back to Drive if it is mounted - more reliable than a browser download.
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy("/content/parity_t4_results.zip", "/content/drive/MyDrive/parity_t4_results.zip")
    print("copied to Drive: MyDrive/parity_t4_results.zip")
else:
    print("Drive not mounted - using browser download instead.")
    try:
        from google.colab import files
        files.download("/content/parity_t4_results.zip")
    except Exception as e:
        print("Download unavailable, grab it from the file browser on the left:", e)

## What to report back

Short version — these are the answers that change what happens next:

1. **Phase 4:** did `-k cuda` pass on real hardware? Any assignment mismatch is the FMA /
   last-block issue, not a flake.
2. **Phase 4:** measured regs/thread, and 100% or 75% occupancy?
3. **Phase 4:** the N=100000 speedup row — never measured before.
4. **Phase 5:** any `wee_meas != wee` mismatches? And at dense, does measured time still
   favour w=0 by ~13%? That is the open Pareto question.
5. **Phase 6:** `gemm_bitexact` all True? And does the 50k GEMM beat fp32?
6. **OpenACC:** host build only, or did you run the HPC SDK cell?